In [1]:
import pandas as pd
import psycopg2

In [2]:
caminho_desafio_dw = 'C:\\Users\\PC\\Documents\\Desafio_dw_eletronicos\\dados\\desafio2_devolucoes.csv'
df_desafio_dw_devolucoes = pd.read_csv(caminho_desafio_dw, sep=',')
df_desafio_dw_devolucoes.head()

,devolucao_id,pedido_id,data_devolucao,cliente_id,produto_id,quantidade_devolvida,valor_devolvido,motivo_devolucao,status_devolucao
0,1,18,2025-06-04,1450,105,1,1031.41,Entrega em atraso,Concluída
1,2,29,2025-05-20,1275,102,1,3268.35,Arrependimento da compra,Concluída
2,3,40,2025-08-20,1068,106,2,3312.61,Produto com defeito,Em análise
3,4,46,2025-11-22,1145,102,1,3676.72,Arrependimento da compra,Em análise
4,5,91,2025-09-27,1127,111,1,1280.05,Arrependimento da compra,Concluída


In [3]:
#Verificar informações da tabela
df_desafio_dw_devolucoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 210 entries, 0 to 209
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   devolucao_id          210 non-null    int64  
 1   pedido_id             210 non-null    int64  
 2   data_devolucao        210 non-null    str    
 3   cliente_id            210 non-null    int64  
 4   produto_id            210 non-null    int64  
 5   quantidade_devolvida  210 non-null    int64  
 6   valor_devolvido       210 non-null    float64
 7   motivo_devolucao      210 non-null    str    
 8   status_devolucao      210 non-null    str    
dtypes: float64(1), int64(5), str(3)
memory usage: 14.9 KB


In [4]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

## CRIANDO TABELA raw_vendas NO BANCO DE DADOS
cursor.execute(""" 
               
                CREATE TABLE IF NOT EXISTS raw.raw_devolucoes
                    (
                        devolucao_id integer,
                        pedido_id integer,
                        data_devolucao date,
                        cliente_id integer,
                        produto_id integer,
                        quantidade_devolvida integer,
                        valor_devolvido decimal(10,2),
                        motivo_devolucao varchar(100),
                        status_devolucao varchar(20)
                    );

               """)
conexao.commit()
cursor.close()
conexao.close()

In [5]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname,user=user,password=password,host=host,port=port)
cursor = conexao.cursor()

##cursor.execute('delete from raw.raw_devolucoes')

for i, df_desafio_dw_devolucoes_raw in df_desafio_dw_devolucoes.iterrows():
    cursor.execute(""" insert into raw.raw_devolucoes(
                    devolucao_id,
	                pedido_id,
	                data_devolucao,
	                cliente_id,
	                produto_id,
	                quantidade_devolvida,
	                valor_devolvido,
	                motivo_devolucao,
	                status_devolucao
                   ) values (%s,%s,%s,%s,%s,%s,%s,%s,%s)
                        
                   """ ,(
                       df_desafio_dw_devolucoes_raw['devolucao_id'],
                       df_desafio_dw_devolucoes_raw['pedido_id'],
                       df_desafio_dw_devolucoes_raw['data_devolucao'],
                       df_desafio_dw_devolucoes_raw['cliente_id'],
                       df_desafio_dw_devolucoes_raw['produto_id'],
                       df_desafio_dw_devolucoes_raw['quantidade_devolvida'],
                       df_desafio_dw_devolucoes_raw['valor_devolvido'],
                       df_desafio_dw_devolucoes_raw['motivo_devolucao'],
                       df_desafio_dw_devolucoes_raw['status_devolucao']
                   ))
conexao.commit()
cursor.close()
conexao.close()

In [6]:
dbname = 'dw_eletronicos'
user = 'postgres'
password = '12345'
host = 'localhost'
port = '5432'

conexao = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
cursor = conexao.cursor()

cursor.execute("""

                CREATE TABLE IF NOT EXISTS staging.stg_devolucoes as
                     select distinct 
                        devolucao_id,
                        pedido_id ,
                        data_devolucao,
                        cliente_id,
                        produto_id,
                        quantidade_devolvida,
                        valor_devolvido :: numeric(10,2),
                        upper(motivo_devolucao) as motivo_devolucao,
                        upper(status_devolucao) as status_devolucao
                    from raw.raw_devolucoes
                    Where valor_devolvido is not null;

                              
               """)
conexao.commit()
cursor.close()
conexao.close()